# 22 condition pseudobulk

**Purpose:** Run condition-level and within-state malignant pseudobulk analyses.

**Original analysis notebook:** `21_condition_specific_malignant_pseudobulk_DE_HVG8000_r0_6_v3_fixed_duplicate_state_column.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 40 — Condition-specific pseudobulk DE in malignant-like compartment

**Goal:** compare gene expression programs between disease conditions (`GBM`, `LGG`, `ndGBM`, `rGBM`) inside the malignant-like compartment.

This notebook performs sample-level pseudobulk analysis to avoid treating cells from the same sample as independent observations.

Main analyses:
1. Overall malignant/cycling compartment pseudobulk DE by condition.
2. Within-state condition DE when enough sample-level pseudobulks are available.
3. Malignant program score comparisons by condition.
4. TSV summaries and simple figures for thesis reporting.

**Interpretation note:** overall malignant-compartment condition DE may reflect both gene-expression shifts and malignant-state composition differences. Within-state comparisons and program scores are included as sensitivity/interpretation aids.


**v2 fix:** pseudobulk aggregation now retains non-group metadata columns such as `condition`, `patient_id`, and `dataset_id`, fixing `KeyError: condition` during condition-specific malignant pseudobulk DE.

In [ ]:

from pathlib import Path
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy import stats
import scanpy as sc
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures" / "condition_malignant_pseudobulk_DE_HVG8000_r0_6"

for d in [METADATA_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation_malignant_states.h5ad"
if not INPUT_H5AD.exists():
    INPUT_H5AD = PROCESSED_DIR / "GBM_core_4datasets_final_annotation.h5ad"

OUT_PREFIX = "condition_malignant_pseudobulk_DE_HVG8000_r0_6"

print("INPUT_H5AD:", INPUT_H5AD)
print("METADATA_DIR:", METADATA_DIR)
print("FIGURES_DIR:", FIGURES_DIR)


In [ ]:

def write_tsv_replace(df, path, index=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=index)
    print(f"[SAVED] {path} shape={df.shape}")

def choose_first_existing_column(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Could not find {label}. Tried: {candidates}")
    return None

def ensure_csr(X):
    if sp.issparse(X):
        return X.tocsr()
    return sp.csr_matrix(X)

def benjamini_hochberg(pvalues):
    p = np.asarray(pvalues, dtype=float)
    q = np.full(p.shape, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return q
    pv = p[valid]
    n = len(pv)
    order = np.argsort(pv)
    ranked = pv[order]
    adj = ranked * n / (np.arange(n) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.minimum(adj, 1.0)
    q_valid = np.empty(n, dtype=float)
    q_valid[order] = adj
    q[valid] = q_valid
    return q

def safe_ttest_ind(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) < 2 or len(y) < 2:
        return np.nan
    if np.nanstd(x) == 0 and np.nanstd(y) == 0:
        return 1.0 if np.nanmean(x) == np.nanmean(y) else np.nan
    try:
        return stats.ttest_ind(x, y, equal_var=False, nan_policy="omit").pvalue
    except Exception:
        return np.nan

def safe_mannwhitney(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    if len(x) < 2 or len(y) < 2:
        return np.nan
    try:
        return stats.mannwhitneyu(x, y, alternative="two-sided").pvalue
    except Exception:
        return np.nan

def safe_kruskal(groups):
    clean = []
    for g in groups:
        g = np.asarray(g, dtype=float)
        g = g[np.isfinite(g)]
        if len(g) > 0:
            clean.append(g)
    if len(clean) < 2:
        return np.nan
    try:
        return stats.kruskal(*clean).pvalue
    except Exception:
        return np.nan

def get_counts_matrix(adata):
    if "counts" in adata.layers:
        return ensure_csr(adata.layers["counts"]), "layers['counts']"
    return ensure_csr(adata.X), "adata.X"

def looks_integer_like_sparse(X, n_rows=1000, n_cols=1000):
    sub = X[:min(n_rows, X.shape[0]), :min(n_cols, X.shape[1])]
    vals = sub.data if sp.issparse(sub) else np.asarray(sub).ravel()
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        return True
    return bool(np.allclose(vals, np.round(vals)))

def aggregate_sparse_by_groups(X, obs_df, group_cols, genes, meta_cols=None):
    """Aggregate a sparse cell x gene matrix to pseudobulk groups.

    group_cols define the aggregation key.
    meta_cols are copied from the first cell in each group. If meta_cols is None,
    all columns in obs_df are retained. This is important when grouping by sample_id
    but still needing condition/patient/dataset metadata downstream.
    """
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    group_cols = list(group_cols)

    if meta_cols is None:
        meta_cols = list(obs_df.columns)
    else:
        meta_cols = list(dict.fromkeys(list(group_cols) + list(meta_cols)))

    missing_group = [c for c in group_cols if c not in obs_df.columns]
    if missing_group:
        raise KeyError(f"Missing group_cols in obs_df: {missing_group}; available={list(obs_df.columns)}")

    meta_cols = [c for c in meta_cols if c in obs_df.columns]
    group_key = obs_df[group_cols].astype(str).agg("||".join, axis=1)
    unique_keys = pd.Index(group_key.unique())
    rows = []
    meta_rows = []
    for key in unique_keys:
        idx = np.where(group_key.values == key)[0]
        s = np.asarray(X[idx, :].sum(axis=0)).ravel()
        rows.append(s)
        meta = {col: obs_df.iloc[idx[0]][col] for col in meta_cols}
        meta["group_key"] = key
        meta["n_cells"] = int(len(idx))
        meta["library_size"] = float(s.sum())
        meta["n_genes_detected"] = int((s > 0).sum())
        meta_rows.append(meta)
    counts_df = pd.DataFrame(np.vstack(rows), index=unique_keys, columns=genes)
    meta_df = pd.DataFrame(meta_rows).set_index("group_key").loc[unique_keys]
    return counts_df, meta_df

def counts_to_logcpm(counts_df):
    counts = counts_df.to_numpy(dtype=float)
    lib = counts.sum(axis=1)
    lib[lib == 0] = np.nan
    cpm = counts / lib[:, None] * 1e6
    return pd.DataFrame(np.log2(cpm + 1.0), index=counts_df.index, columns=counts_df.columns)

def filter_genes_by_cpm(counts_df, min_cpm=1.0, min_samples=3):
    counts = counts_df.to_numpy(dtype=float)
    lib = counts.sum(axis=1)
    lib[lib == 0] = np.nan
    cpm = counts / lib[:, None] * 1e6
    keep = np.nansum(cpm >= min_cpm, axis=0) >= min_samples
    return pd.Series(keep, index=counts_df.columns)


In [ ]:

def run_global_kruskal(logcpm_df, meta_df, condition_col, genes):
    conditions = sorted(meta_df[condition_col].dropna().astype(str).unique())
    rows = []
    for gene in genes:
        groups = [logcpm_df.loc[meta_df[condition_col].astype(str) == cond, gene].values for cond in conditions]
        p = safe_kruskal(groups)
        means = {f"mean_logcpm_{cond}": float(np.nanmean(g)) if len(g) else np.nan for cond, g in zip(conditions, groups)}
        rows.append({"gene": gene, "test": "Kruskal-Wallis", "n_conditions": len(conditions), "pvalue": p, **means})
    df = pd.DataFrame(rows)
    df["padj"] = benjamini_hochberg(df["pvalue"].values)
    return df.sort_values(["padj", "pvalue"], na_position="last")

def run_pairwise_de(logcpm_df, meta_df, condition_col, genes, min_samples_per_group=3):
    conditions = sorted(meta_df[condition_col].dropna().astype(str).unique())
    contrasts = []
    for i in range(len(conditions)):
        for j in range(i + 1, len(conditions)):
            contrasts.append((conditions[i], conditions[j]))
    all_rows = []
    summary_rows = []
    for cond_a, cond_b in contrasts:
        mask_a = meta_df[condition_col].astype(str) == cond_a
        mask_b = meta_df[condition_col].astype(str) == cond_b
        n_a, n_b = int(mask_a.sum()), int(mask_b.sum())
        contrast = f"{cond_a}_vs_{cond_b}"
        if n_a < min_samples_per_group or n_b < min_samples_per_group:
            summary_rows.append({
                "contrast": contrast, "condition_A": cond_a, "condition_B": cond_b,
                "n_A": n_a, "n_B": n_b, "status": "skipped_low_sample_count"
            })
            continue
        rows = []
        for gene in genes:
            xa = logcpm_df.loc[mask_a, gene].values
            xb = logcpm_df.loc[mask_b, gene].values
            mean_a = float(np.nanmean(xa))
            mean_b = float(np.nanmean(xb))
            rows.append({
                "contrast": contrast,
                "condition_A": cond_a,
                "condition_B": cond_b,
                "gene": gene,
                "n_A": n_a,
                "n_B": n_b,
                "mean_logcpm_A": mean_a,
                "mean_logcpm_B": mean_b,
                "log2FC_A_vs_B": mean_a - mean_b,
                "pvalue_welch": safe_ttest_ind(xa, xb),
                "pvalue_mannwhitney": safe_mannwhitney(xa, xb),
            })
        df = pd.DataFrame(rows)
        df["padj_welch"] = benjamini_hochberg(df["pvalue_welch"].values)
        df["padj_mannwhitney"] = benjamini_hochberg(df["pvalue_mannwhitney"].values)
        all_rows.append(df)
        summary_rows.append({
            "contrast": contrast,
            "condition_A": cond_a,
            "condition_B": cond_b,
            "n_A": n_a,
            "n_B": n_b,
            "status": "tested",
            "n_genes_tested": int(len(genes)),
            "n_sig_welch_fdr_0_05": int((df["padj_welch"] < 0.05).sum()),
            "n_sig_mw_fdr_0_05": int((df["padj_mannwhitney"] < 0.05).sum()),
            "n_sig_abs_log2fc_0_5_fdr_0_05": int(((df["padj_welch"] < 0.05) & (df["log2FC_A_vs_B"].abs() >= 0.5)).sum()),
        })
    res = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    summary = pd.DataFrame(summary_rows)
    return res, summary

def extract_top_up_down(de_df, n_top=100, padj_col="padj_welch", lfc_col="log2FC_A_vs_B"):
    rows = []
    for contrast, sub in de_df.groupby("contrast"):
        sub = sub.copy().sort_values([padj_col, "pvalue_welch"], na_position="last")
        up = sub[sub[lfc_col] > 0].head(n_top).copy()
        up["direction"] = "up_in_A"
        down = sub[sub[lfc_col] < 0].head(n_top).copy()
        down["direction"] = "up_in_B"
        rows.extend([up, down])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


In [ ]:

adata = sc.read_h5ad(INPUT_H5AD)
print(adata)

obs = adata.obs.copy()
genes = pd.Index(adata.var_names.astype(str))
X_counts, counts_source = get_counts_matrix(adata)

condition_col = choose_first_existing_column(obs, ["condition", "disease_condition", "diagnosis", "group"], label="condition column")
sample_col = choose_first_existing_column(obs, ["sample_id", "sample", "sample_name", "orig.ident", "library_id"], label="sample column")
patient_col = choose_first_existing_column(obs, ["patient_id", "patient", "case_id", "donor_id"], required=False, label="patient column")
dataset_col = choose_first_existing_column(obs, ["core_dataset", "dataset_id", "dataset", "study_id"], required=False, label="dataset column")
cluster_col = choose_first_existing_column(obs, ["leiden_scvi_main", "leiden_scvi_HVG8000_r0_6", "leiden"], label="cluster column")

print("condition_col:", condition_col)
print("sample_col:", sample_col)
print("patient_col:", patient_col)
print("dataset_col:", dataset_col)
print("cluster_col:", cluster_col)
print("counts_source:", counts_source)
print("counts integer-like:", looks_integer_like_sparse(X_counts))

readiness = pd.DataFrame([
    ["input_h5ad", str(INPUT_H5AD)],
    ["n_cells", adata.n_obs],
    ["n_genes", adata.n_vars],
    ["counts_source", counts_source],
    ["counts_integer_like_sample", looks_integer_like_sparse(X_counts)],
    ["condition_col", condition_col],
    ["sample_col", sample_col],
    ["patient_col", patient_col if patient_col else ""],
    ["dataset_col", dataset_col if dataset_col else ""],
    ["cluster_col", cluster_col],
], columns=["item", "value"])

write_tsv_replace(readiness, METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv")


In [ ]:

MALIGNANT_STATE_BY_CLUSTER = {
    "0": "OPC_NPC_like_malignant",
    "4": "hypoxia_stress_malignant",
    "7": "AC_like_reactive_malignant",
    "11": "NPC_neuronal_like_malignant",
    "2": "cycling_malignant_candidate",
}

obs["cluster_for_malignant_state"] = obs[cluster_col].astype(str)
obs["condition_malignant_state"] = obs["cluster_for_malignant_state"].map(MALIGNANT_STATE_BY_CLUSTER).fillna("not_malignant_state")

malig_mask = obs["condition_malignant_state"] != "not_malignant_state"
print("Malignant/cycling cells selected:", int(malig_mask.sum()))

state_counts = (
    obs.loc[malig_mask, "condition_malignant_state"]
    .value_counts()
    .rename_axis("condition_malignant_state")
    .reset_index(name="n_cells")
)
state_counts["fraction_of_selected"] = state_counts["n_cells"] / state_counts["n_cells"].sum()
write_tsv_replace(state_counts, METADATA_DIR / f"{OUT_PREFIX}_state_cell_counts.tsv")
display(state_counts)


In [ ]:

obs_sel = obs.loc[malig_mask].copy()
X_sel = X_counts[malig_mask.values, :]

meta_cols_to_keep = [condition_col]
if patient_col:
    meta_cols_to_keep.append(patient_col)
if dataset_col:
    meta_cols_to_keep.append(dataset_col)

pb_counts, pb_meta = aggregate_sparse_by_groups(X_sel, obs_sel[[sample_col] + meta_cols_to_keep], sample_col, genes)

pb_meta = pb_meta.rename(columns={sample_col: "sample_id", condition_col: "condition"})
if patient_col:
    pb_meta = pb_meta.rename(columns={patient_col: "patient_id"})
if dataset_col:
    pb_meta = pb_meta.rename(columns={dataset_col: "dataset_id"})

pb_meta["condition"] = pb_meta["condition"].astype(str)
pb_meta["sample_id"] = pb_meta["sample_id"].astype(str)

SAMPLE_MIN_CELLS = 30
keep_samples = pb_meta["n_cells"] >= SAMPLE_MIN_CELLS
pb_counts_f = pb_counts.loc[keep_samples].copy()
pb_meta_f = pb_meta.loc[keep_samples].copy()

gene_keep = filter_genes_by_cpm(pb_counts_f, min_cpm=1.0, min_samples=3)
genes_test = gene_keep[gene_keep].index.tolist()

print("Pseudobulk samples:", pb_counts.shape[0])
print("Pseudobulk samples passing:", pb_counts_f.shape[0])
print("Genes passing expression filter:", len(genes_test))

sample_qc = pb_meta_f.reset_index().rename(columns={"index": "group_key"})
sample_qc["n_genes_total"] = adata.n_vars
sample_qc["n_genes_tested"] = len(genes_test)
write_tsv_replace(sample_qc, METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_sample_QC.tsv")

pb_counts_f[genes_test].to_csv(METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_counts_pseudobulk.tsv.gz", sep="\t")
logcpm_overall = counts_to_logcpm(pb_counts_f[genes_test])
logcpm_overall.to_csv(METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_logCPM_pseudobulk.tsv.gz", sep="\t")

condition_summary = (
    pb_meta_f.groupby("condition")
    .agg(
        n_samples=("sample_id", "nunique"),
        n_pseudobulks=("sample_id", "size"),
        median_cells=("n_cells", "median"),
        total_cells=("n_cells", "sum"),
        median_library_size=("library_size", "median"),
    )
    .reset_index()
)
write_tsv_replace(condition_summary, METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_condition_summary.tsv")
display(condition_summary)


In [ ]:

global_kruskal = run_global_kruskal(logcpm_overall, pb_meta_f, "condition", genes_test)
write_tsv_replace(global_kruskal, METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_global_kruskal.tsv")

pairwise_de, pairwise_summary = run_pairwise_de(logcpm_overall, pb_meta_f, "condition", genes_test, min_samples_per_group=3)
write_tsv_replace(pairwise_summary, METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_test_summary.tsv")
write_tsv_replace(pairwise_de, METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_all_genes.tsv")

top100_overall = extract_top_up_down(pairwise_de, n_top=100)
write_tsv_replace(top100_overall, METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_top100_up_down.tsv")

sig_overall = pairwise_de[
    (pairwise_de["padj_welch"] < 0.05) &
    (pairwise_de["log2FC_A_vs_B"].abs() >= 0.5)
].copy()
write_tsv_replace(sig_overall, METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_significant_FDR0_05_absLFC0_5.tsv")

display(pairwise_summary)
display(top100_overall.head(20))


In [ ]:

obs_state = obs_sel.copy()
obs_state["condition_malignant_state"] = obs_state["condition_malignant_state"].astype(str)

state_group_cols = [sample_col, "condition_malignant_state"]
state_meta_cols = [sample_col, "condition_malignant_state", condition_col]
if patient_col:
    state_meta_cols.append(patient_col)
if dataset_col:
    state_meta_cols.append(dataset_col)

state_counts, state_meta = aggregate_sparse_by_groups(X_sel, obs_state[state_meta_cols], state_group_cols, genes)

state_meta = state_meta.rename(columns={sample_col: "sample_id", condition_col: "condition"})
if patient_col:
    state_meta = state_meta.rename(columns={patient_col: "patient_id"})
if dataset_col:
    state_meta = state_meta.rename(columns={dataset_col: "dataset_id"})

MIN_CELLS_PER_SAMPLE_STATE = 30
keep_state_pb = state_meta["n_cells"] >= MIN_CELLS_PER_SAMPLE_STATE
state_counts_f = state_counts.loc[keep_state_pb].copy()
state_meta_f = state_meta.loc[keep_state_pb].copy()

state_gene_keep = filter_genes_by_cpm(state_counts_f, min_cpm=1.0, min_samples=3)
state_genes_test = state_gene_keep[state_gene_keep].index.tolist()

logcpm_state = counts_to_logcpm(state_counts_f[state_genes_test])

state_pb_qc = state_meta_f.reset_index().rename(columns={"index": "group_key"})
state_pb_qc["n_genes_tested"] = len(state_genes_test)
write_tsv_replace(state_pb_qc, METADATA_DIR / f"{OUT_PREFIX}_sample_state_pseudobulk_QC.tsv")

state_condition_summary = (
    state_meta_f.groupby(["condition_malignant_state", "condition"])
    .agg(
        n_pseudobulks=("sample_id", "size"),
        n_samples=("sample_id", "nunique"),
        total_cells=("n_cells", "sum"),
        median_cells=("n_cells", "median"),
    )
    .reset_index()
)
write_tsv_replace(state_condition_summary, METADATA_DIR / f"{OUT_PREFIX}_state_condition_pseudobulk_summary.tsv")
display(state_condition_summary)


In [ ]:

within_rows = []
within_summary_rows = []
top_rows = []

# Robust helper: add condition_malignant_state only if it is not already present.
def add_state_column(df, state, col="condition_malignant_state"):
    df = df.copy()
    if len(df) == 0:
        return df
    if col in df.columns:
        # Keep the existing column, but fill missing/empty values if needed.
        df[col] = df[col].astype(str)
        empty_mask = df[col].isin(["", "nan", "None", "NaN"])
        if empty_mask.any():
            df.loc[empty_mask, col] = str(state)
    else:
        df.insert(0, col, str(state))
    return df

for state, idx in state_meta_f.groupby("condition_malignant_state").groups.items():
    state_idx = list(idx)
    meta_s = state_meta_f.loc[state_idx].copy()
    log_s = logcpm_state.loc[state_idx, :].copy()

    de_s, summary_s = run_pairwise_de(log_s, meta_s, "condition", state_genes_test, min_samples_per_group=3)

    if len(summary_s) > 0:
        summary_s = add_state_column(summary_s, state)
        within_summary_rows.append(summary_s)

    if len(de_s) > 0:
        de_s = add_state_column(de_s, state)
        within_rows.append(de_s)
        top_s = extract_top_up_down(de_s, n_top=50)
        if len(top_s) > 0:
            top_s = add_state_column(top_s, state)
            top_rows.append(top_s)

within_state_de = pd.concat(within_rows, ignore_index=True) if within_rows else pd.DataFrame()
within_state_summary = pd.concat(within_summary_rows, ignore_index=True) if within_summary_rows else pd.DataFrame()
within_state_top50 = pd.concat(top_rows, ignore_index=True) if top_rows else pd.DataFrame()

write_tsv_replace(within_state_summary, METADATA_DIR / f"{OUT_PREFIX}_within_state_condition_DE_test_summary.tsv")
write_tsv_replace(within_state_de, METADATA_DIR / f"{OUT_PREFIX}_within_state_condition_DE_all_genes.tsv")
write_tsv_replace(within_state_top50, METADATA_DIR / f"{OUT_PREFIX}_within_state_condition_DE_top50_up_down.tsv")

display(within_state_summary.head(30))
display(within_state_top50.head(20))


In [ ]:

PROGRAMS = {
    "OPC_proneural": ["OLIG1", "OLIG2", "SOX10", "PDGFRA", "BCAN", "PTPRZ1", "PLP1", "CSPG4", "DLL3"],
    "NPC_neural": ["SOX11", "DCX", "STMN2", "TUBB3", "RBFOX2", "ELAVL4", "DLX5", "MYT1L", "SLC17A6"],
    "AC_reactive": ["GFAP", "AQP4", "CLU", "SPARCL1", "ID3", "ID4", "FAM107A", "SLC1A3"],
    "MES_like": ["CHI3L1", "CD44", "VIM", "TGFBI", "SERPINE1", "ANXA1", "LGALS3", "FN1", "CTSB"],
    "hypoxia_stress": ["NDRG1", "VEGFA", "SLC2A1", "HILPDA", "ADM", "EGLN3", "P4HA1", "PGK1", "CA9"],
    "cycling": ["TOP2A", "MKI67", "UBE2C", "PBK", "ASPM", "RRM2", "CENPF", "DLGAP5", "BIRC5"],
    "immune_myeloid_review": ["C1QA", "C1QB", "C1QC", "TYROBP", "LYZ", "CSF1R", "HLA-DRA", "CD74", "AIF1"],
}

program_presence_rows = []
z_log = logcpm_overall.copy()
z_log = z_log.sub(z_log.mean(axis=0), axis=1)
z_log = z_log.div(z_log.std(axis=0).replace(0, np.nan), axis=1)

program_scores = pb_meta_f[["sample_id", "condition", "n_cells", "library_size"]].copy()

for prog, markers in PROGRAMS.items():
    present = [g for g in markers if g in z_log.columns]
    missing = [g for g in markers if g not in z_log.columns]
    program_presence_rows.append({
        "program": prog,
        "n_markers_total": len(markers),
        "n_markers_present": len(present),
        "n_markers_missing": len(missing),
        "markers_present": ",".join(present),
        "markers_missing": ",".join(missing),
    })
    program_scores[f"{prog}_score"] = z_log[present].mean(axis=1).values if present else np.nan

program_presence = pd.DataFrame(program_presence_rows)
write_tsv_replace(program_presence, METADATA_DIR / f"{OUT_PREFIX}_program_marker_presence.tsv")

write_tsv_replace(
    program_scores.reset_index().rename(columns={"index": "group_key"}),
    METADATA_DIR / f"{OUT_PREFIX}_sample_level_program_scores.tsv"
)

program_test_rows = []
program_pair_rows = []
conditions = sorted(program_scores["condition"].astype(str).unique())

for prog in PROGRAMS.keys():
    score_col = f"{prog}_score"
    groups = [program_scores.loc[program_scores["condition"].astype(str) == cond, score_col].values for cond in conditions]
    p = safe_kruskal(groups)
    program_test_rows.append({
        "program": prog,
        "score_col": score_col,
        "test": "Kruskal-Wallis",
        "pvalue": p,
        **{f"median_{cond}": float(np.nanmedian(g)) if len(g) else np.nan for cond, g in zip(conditions, groups)}
    })
    for i in range(len(conditions)):
        for j in range(i + 1, len(conditions)):
            a, b = conditions[i], conditions[j]
            xa = program_scores.loc[program_scores["condition"].astype(str) == a, score_col].values
            xb = program_scores.loc[program_scores["condition"].astype(str) == b, score_col].values
            program_pair_rows.append({
                "program": prog,
                "condition_A": a,
                "condition_B": b,
                "n_A": int(np.isfinite(xa).sum()),
                "n_B": int(np.isfinite(xb).sum()),
                "median_A": float(np.nanmedian(xa)) if len(xa) else np.nan,
                "median_B": float(np.nanmedian(xb)) if len(xb) else np.nan,
                "delta_median_A_minus_B": float(np.nanmedian(xa) - np.nanmedian(xb)) if len(xa) and len(xb) else np.nan,
                "pvalue_mannwhitney": safe_mannwhitney(xa, xb),
            })

program_global = pd.DataFrame(program_test_rows)
program_global["padj"] = benjamini_hochberg(program_global["pvalue"].values)
program_global = program_global.sort_values(["padj", "pvalue"], na_position="last")

program_pairwise = pd.DataFrame(program_pair_rows)
program_pairwise["padj_mannwhitney"] = benjamini_hochberg(program_pairwise["pvalue_mannwhitney"].values)
program_pairwise = program_pairwise.sort_values(["padj_mannwhitney", "pvalue_mannwhitney"], na_position="last")

write_tsv_replace(program_global, METADATA_DIR / f"{OUT_PREFIX}_program_score_global_kruskal.tsv")
write_tsv_replace(program_pairwise, METADATA_DIR / f"{OUT_PREFIX}_program_score_pairwise_mannwhitney.tsv")

display(program_global)
display(program_pairwise.head(20))


In [ ]:

figure_rows = []

def save_fig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    figure_rows.append({"figure": path.name, "path": str(path)})

plot_df = condition_summary.copy()
x = np.arange(len(plot_df))
plt.figure(figsize=(8, 4))
plt.bar(x, plot_df["n_samples"].values)
plt.xticks(x, plot_df["condition"].values, rotation=45, ha="right")
plt.ylabel("Number of samples")
plt.title("Pseudobulk samples per condition")
save_fig(FIGURES_DIR / f"{OUT_PREFIX}_condition_pseudobulk_sample_counts.png")

score_cols = [f"{p}_score" for p in PROGRAMS.keys()]
heat = program_scores.groupby("condition")[score_cols].median()
plt.figure(figsize=(10, max(3, 0.4 * len(heat))))
plt.imshow(heat.values, aspect="auto")
plt.colorbar(label="Median z-score")
plt.yticks(np.arange(heat.shape[0]), heat.index)
plt.xticks(np.arange(heat.shape[1]), [c.replace("_score", "") for c in heat.columns], rotation=45, ha="right")
plt.title("Malignant-compartment program scores by condition")
save_fig(FIGURES_DIR / f"{OUT_PREFIX}_program_score_condition_medians.png")

top_genes = global_kruskal.head(30)["gene"].tolist()
if len(top_genes) > 0:
    heat2 = logcpm_overall[top_genes].copy()
    order = pb_meta_f.sort_values(["condition", "sample_id"]).index
    heat2 = heat2.loc[order]
    heat2 = heat2.sub(heat2.mean(axis=0), axis=1).div(heat2.std(axis=0).replace(0, np.nan), axis=1)
    heat2 = heat2.clip(-3, 3)
    plt.figure(figsize=(10, 8))
    plt.imshow(heat2.T.values, aspect="auto", vmin=-3, vmax=3)
    plt.colorbar(label="z(logCPM)")
    plt.yticks(np.arange(len(top_genes)), top_genes)
    plt.xticks([])
    plt.title("Top condition-associated genes in malignant/cycling pseudobulk")
    save_fig(FIGURES_DIR / f"{OUT_PREFIX}_top30_global_condition_DE_heatmap.png")

figure_manifest = pd.DataFrame(figure_rows)
write_tsv_replace(figure_manifest, METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv")
display(figure_manifest)


In [ ]:

expected_outputs = {
    "readiness": METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv",
    "state_cell_counts": METADATA_DIR / f"{OUT_PREFIX}_state_cell_counts.tsv",
    "overall_malignant_sample_QC": METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_sample_QC.tsv",
    "overall_malignant_condition_summary": METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_condition_summary.tsv",
    "overall_malignant_counts_pseudobulk": METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_counts_pseudobulk.tsv.gz",
    "overall_malignant_logCPM_pseudobulk": METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_logCPM_pseudobulk.tsv.gz",
    "overall_malignant_global_kruskal": METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_global_kruskal.tsv",
    "overall_malignant_pairwise_DE_test_summary": METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_test_summary.tsv",
    "overall_malignant_pairwise_DE_all_genes": METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_all_genes.tsv",
    "overall_malignant_pairwise_DE_top100_up_down": METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_top100_up_down.tsv",
    "overall_malignant_pairwise_DE_significant": METADATA_DIR / f"{OUT_PREFIX}_overall_malignant_pairwise_DE_significant_FDR0_05_absLFC0_5.tsv",
    "sample_state_pseudobulk_QC": METADATA_DIR / f"{OUT_PREFIX}_sample_state_pseudobulk_QC.tsv",
    "state_condition_pseudobulk_summary": METADATA_DIR / f"{OUT_PREFIX}_state_condition_pseudobulk_summary.tsv",
    "within_state_condition_DE_test_summary": METADATA_DIR / f"{OUT_PREFIX}_within_state_condition_DE_test_summary.tsv",
    "within_state_condition_DE_all_genes": METADATA_DIR / f"{OUT_PREFIX}_within_state_condition_DE_all_genes.tsv",
    "within_state_condition_DE_top50_up_down": METADATA_DIR / f"{OUT_PREFIX}_within_state_condition_DE_top50_up_down.tsv",
    "program_marker_presence": METADATA_DIR / f"{OUT_PREFIX}_program_marker_presence.tsv",
    "sample_level_program_scores": METADATA_DIR / f"{OUT_PREFIX}_sample_level_program_scores.tsv",
    "program_score_global_kruskal": METADATA_DIR / f"{OUT_PREFIX}_program_score_global_kruskal.tsv",
    "program_score_pairwise_mannwhitney": METADATA_DIR / f"{OUT_PREFIX}_program_score_pairwise_mannwhitney.tsv",
    "figure_manifest": METADATA_DIR / f"{OUT_PREFIX}_figure_manifest.tsv",
}

manifest = pd.DataFrame([
    {
        "output_name": k,
        "path": str(v),
        "exists": Path(v).exists(),
        "size_MB": round(Path(v).stat().st_size / 1024 / 1024, 3) if Path(v).exists() else np.nan
    }
    for k, v in expected_outputs.items()
])

write_tsv_replace(manifest, METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv")
display(manifest)

print("[DONE] Condition-specific pseudobulk DE in malignant-like compartment.")
